# wattnet-client — quickstart with real data

This notebook exercises every method of `wattnet.client.WattnetClient` against the
**live, production** wattnet API (`https://api.wattnet.eu`) and keeps the real
responses as saved output, so you can see exactly what each call returns without
needing your own credentials just to read it.

To re-run this notebook yourself:

1. Get a bearer token (a pre-issued one, or via `email`/`password` — see below).
2. Set it as an environment variable, **never hardcode it in the notebook**:

   ```bash
   export WATTNET_TOKEN="<your-token>"
   jupyter nbconvert --to notebook --execute --inplace examples/quickstart.ipynb
   ```

In [1]:
import os

from wattnet.client import WattnetClient, WattnetRawClient

client = WattnetClient(token=os.environ["WATTNET_TOKEN"])
client

WattnetClient(base_url='https://api.wattnet.eu/v1', auth=StaticTokenProvider)

Instead of a static token, you can also authenticate with an email/password pair
already registered with the wattnet token-request service — the client fetches and
caches a token for you (refreshing it automatically before it expires):

```python
client = WattnetClient(email="you@example.com", password="<your-password>")
```

We'll use a fixed 6-hour window for every time-series call below, so the examples
are reproducible and the tables stay a readable size.

In [2]:
START = "2026-07-15T00:00:00Z"
END = "2026-07-15T06:00:00Z"
ZONE = "ES"

## Zones

`get_zones()` — metadata and neighbours for every supported zone.

In [3]:
zones = client.get_zones()
print(zones.shape)
zones.head()

(60, 7)


,zone,full_name,eic_code,country_code,country_name,provider,neighbours
0,AL,Albania,10YAL-KESH-----5,ALB,Albania,ENTSO-E,"[GR, ME, RS, XK]"
1,AM,Armenia,10Y1001A1001B004,ARM,Armenia,ENTSO-E,[GE]
2,AT,Austria,10YAT-APG------L,AUT,Austria,ENTSO-E,"[CH, CZ, DE, HU, IT_NORTH, SI]"
3,AZ,Azerbaijan,10Y1001A1001B05V,AZE,Azerbaijan,ENTSO-E,[GE]
4,BA,Bosnia and Herzegovina,10YBA-JPCC-----D,BIH,Bosnia and Herzegovina,ENTSO-E,"[HR, ME, RS]"


## Energy metrics

### Generation

`get_generation()` — electricity generation by production type.

In [4]:
generation = client.get_generation(zone=ZONE, start=START, end=END)
print(generation.shape)
generation.head()

(384, 9)


,zone,unit,valid,zone_status,production_type,data_state,datasource,timestamp,value
0,ES,MW,True,complete,biomass,official,ENTSO-E,2026-07-15 00:00:00+00:00,516.0
1,ES,MW,True,complete,biomass,official,ENTSO-E,2026-07-15 00:15:00+00:00,516.0
2,ES,MW,True,complete,biomass,official,ENTSO-E,2026-07-15 00:30:00+00:00,516.0
3,ES,MW,True,complete,biomass,official,ENTSO-E,2026-07-15 00:45:00+00:00,512.0
4,ES,MW,True,complete,biomass,official,ENTSO-E,2026-07-15 01:00:00+00:00,512.0


### Load

`get_load()` — total electricity demand.

In [5]:
load = client.get_load(zone=ZONE, start=START, end=END)
print(load.shape)
load.head()

(24, 8)


,zone,unit,valid,zone_status,data_state,datasource,timestamp,value
0,ES,MW,True,complete,official,ENTSO-E,2026-07-15 00:00:00+00:00,29032.0
1,ES,MW,True,complete,official,ENTSO-E,2026-07-15 00:15:00+00:00,28668.0
2,ES,MW,True,complete,official,ENTSO-E,2026-07-15 00:30:00+00:00,28436.0
3,ES,MW,True,complete,official,ENTSO-E,2026-07-15 00:45:00+00:00,28244.0
4,ES,MW,True,complete,official,ENTSO-E,2026-07-15 01:00:00+00:00,28080.0


### Imports

`get_imports()` — electricity imported into the zone, by origin.

In [6]:
imports = client.get_imports(zone=ZONE, start=START, end=END)
print(imports.shape)
imports.head()

(48, 9)


,zone,unit,valid,zone_status,source,data_state,datasource,timestamp,value
0,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 00:00:00+00:00,601.84
1,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 00:15:00+00:00,646.92
2,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 00:30:00+00:00,527.52
3,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 00:45:00+00:00,522.84
4,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 01:00:00+00:00,507.52


### Exports

`get_exports()` — electricity exported from the zone, by destination.

In [7]:
exports = client.get_exports(zone=ZONE, start=START, end=END)
print(exports.shape)
exports.head()

(48, 9)


,zone,unit,valid,zone_status,destination,data_state,datasource,timestamp,value
0,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 00:00:00+00:00,0.0
1,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 00:15:00+00:00,0.0
2,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 00:30:00+00:00,0.0
3,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 00:45:00+00:00,0.0
4,ES,MW,True,complete,FR,official,ENTSO-E,2026-07-15 01:00:00+00:00,0.0


### Mix

`get_mix()` — flow-traced generation mix by production type.

In [8]:
mix = client.get_mix(zone=ZONE, start=START, end=END)
print(mix.shape)
mix.head()

(384, 9)


,zone,unit,valid,zone_status,production_type,data_state,datasource,timestamp,value
0,ES,MW,True,complete,biomass,official,flow_tracing,2026-07-15 00:00:00+00:00,504.86
1,ES,MW,True,complete,biomass,official,flow_tracing,2026-07-15 00:15:00+00:00,502.14
2,ES,MW,True,complete,biomass,official,flow_tracing,2026-07-15 00:30:00+00:00,500.03
3,ES,MW,True,complete,biomass,official,flow_tracing,2026-07-15 00:45:00+00:00,495.33
4,ES,MW,True,complete,biomass,official,flow_tracing,2026-07-15 01:00:00+00:00,495.36


## Environmental metrics

### Footprints

`get_footprints()` — carbon/water footprint as a time series.

In [9]:
footprints = client.get_footprints(zone=ZONE, start=START, end=END)
print(footprints.shape)
footprints.head()

(48, 9)


,footprint_type,scope,zone,unit,coverage,valid,zone_status,timestamp,value
0,carbon,life-cycle,ES,gCO2/kWh,global,True,complete,2026-07-15 00:00:00+00:00,231.17
1,carbon,life-cycle,ES,gCO2/kWh,global,True,complete,2026-07-15 00:15:00+00:00,231.45
2,carbon,life-cycle,ES,gCO2/kWh,global,True,complete,2026-07-15 00:30:00+00:00,234.46
3,carbon,life-cycle,ES,gCO2/kWh,global,True,complete,2026-07-15 00:45:00+00:00,233.86
4,carbon,life-cycle,ES,gCO2/kWh,global,True,complete,2026-07-15 01:00:00+00:00,237.09


Set `aggregate=True` to get a single value per footprint type over the window instead of a time series:

In [10]:
footprints_agg = client.get_footprints(zone=ZONE, start=START, end=END, aggregate=True)
footprints_agg

,footprint_type,scope,zone,unit,coverage,start,end,value,valid,zone_status,aggregation_method
0,carbon,life-cycle,ES,gCO2/kWh,global,2026-07-15T00:00:00Z,2026-07-15T06:00:00Z,236.71,True,complete,time-weighted-average
1,water,life-cycle,ES,l/kWh,global,2026-07-15T00:00:00Z,2026-07-15T06:00:00Z,8.53,True,complete,time-weighted-average


### Impacts

`get_impacts()` — water impact of electricity consumption.

In [11]:
impacts = client.get_impacts(zone=ZONE, start=START, end=END)
print(impacts.shape)
impacts.head()

(24, 9)


,impact_type,scope,zone,unit,coverage,valid,zone_status,timestamp,value
0,water,operational,ES,stress-l/kWh,global,True,complete,2026-07-15 00:00:00+00:00,2.77
1,water,operational,ES,stress-l/kWh,global,True,complete,2026-07-15 00:15:00+00:00,2.65
2,water,operational,ES,stress-l/kWh,global,True,complete,2026-07-15 00:30:00+00:00,2.58
3,water,operational,ES,stress-l/kWh,global,True,complete,2026-07-15 00:45:00+00:00,2.58
4,water,operational,ES,stress-l/kWh,global,True,complete,2026-07-15 01:00:00+00:00,2.50


### EnvironmentalScore

`get_environmental_score()` — a single 0-100 sustainability score.

In [12]:
environmental_score = client.get_environmental_score(zone=ZONE, start=START, end=END)
print(environmental_score.shape)
environmental_score.head()

(24, 7)


,scope,zone,coverage,valid,zone_status,timestamp,value
0,operational,ES,global,True,complete,2026-07-15 00:00:00+00:00,76.67
1,operational,ES,global,True,complete,2026-07-15 00:15:00+00:00,76.78
2,operational,ES,global,True,complete,2026-07-15 00:30:00+00:00,76.62
3,operational,ES,global,True,complete,2026-07-15 00:45:00+00:00,76.67
4,operational,ES,global,True,complete,2026-07-15 01:00:00+00:00,76.58


## Shares metrics

### Flow share

`get_flow_share()` — share of this zone's production exported to each destination zone.

In [13]:
flow_share = client.get_flow_share(zone=ZONE, start=START, end=END)
print(flow_share.shape)
flow_share.head()

(1248, 7)


,zone,unit,valid,zone_status,destination,timestamp,value
0,ES,%,True,complete,AT,2026-07-15 00:00:00+00:00,0.0
1,ES,%,True,complete,AT,2026-07-15 00:15:00+00:00,0.0
2,ES,%,True,complete,AT,2026-07-15 00:30:00+00:00,0.0
3,ES,%,True,complete,AT,2026-07-15 00:45:00+00:00,0.0
4,ES,%,True,complete,AT,2026-07-15 01:00:00+00:00,0.0


### Mix share

`get_mix_share()` — share of this zone's mix coming from each origin zone.

In [14]:
mix_share = client.get_mix_share(zone=ZONE, start=START, end=END)
print(mix_share.shape)
mix_share.head()

(1248, 7)


,zone,unit,valid,zone_status,origin,timestamp,value
0,ES,%,True,complete,AT,2026-07-15 00:00:00+00:00,0.0
1,ES,%,True,complete,AT,2026-07-15 00:15:00+00:00,0.0
2,ES,%,True,complete,AT,2026-07-15 00:30:00+00:00,0.0
3,ES,%,True,complete,AT,2026-07-15 00:45:00+00:00,0.0
4,ES,%,True,complete,AT,2026-07-15 01:00:00+00:00,0.0


### Footprint share

`get_footprint_share()` — this zone's footprint decomposed by origin zone.

In [15]:
footprint_share = client.get_footprint_share(zone=ZONE, start=START, end=END)
print(footprint_share.shape)
footprint_share.head()

(2496, 9)


,zone,footprint_type,scope,unit,valid,zone_status,source,timestamp,value
0,ES,carbon,life-cycle,gCO2/kWh,True,complete,AT,2026-07-15 00:00:00+00:00,0.0
1,ES,carbon,life-cycle,gCO2/kWh,True,complete,AT,2026-07-15 00:15:00+00:00,0.0
2,ES,carbon,life-cycle,gCO2/kWh,True,complete,AT,2026-07-15 00:30:00+00:00,0.0
3,ES,carbon,life-cycle,gCO2/kWh,True,complete,AT,2026-07-15 00:45:00+00:00,0.0
4,ES,carbon,life-cycle,gCO2/kWh,True,complete,AT,2026-07-15 01:00:00+00:00,0.0


### Impact share

`get_impact_share()` — this zone's impact decomposed by origin zone.

In [16]:
impact_share = client.get_impact_share(zone=ZONE, start=START, end=END)
print(impact_share.shape)
impact_share.head()

(1248, 9)


,zone,impact_type,scope,unit,valid,zone_status,source,timestamp,value
0,ES,water,operational,stress-l/kWh,True,complete,AT,2026-07-15 00:00:00+00:00,0.0
1,ES,water,operational,stress-l/kWh,True,complete,AT,2026-07-15 00:15:00+00:00,0.0
2,ES,water,operational,stress-l/kWh,True,complete,AT,2026-07-15 00:30:00+00:00,0.0
3,ES,water,operational,stress-l/kWh,True,complete,AT,2026-07-15 00:45:00+00:00,0.0
4,ES,water,operational,stress-l/kWh,True,complete,AT,2026-07-15 01:00:00+00:00,0.0


## Factors

`get_factors()` — global emission/consumption factors, independent of any zone.

In [17]:
factors = client.get_factors(factor_type="carbon", production_type="solar")
factors

,factor_type,production_type,scope,unit,source,year,source_link,timestamp,value
0,carbon,solar,life-cycle,gCO2/kWh,UNECE,2022,https://unece.org/sites/default/files/2022-04/...,2026-07-17 12:30:00+00:00,36.95
1,carbon,solar,operational,gCO2/kWh,IPCC,2014,https://www.ipcc.ch/site/assets/uploads/2018/0...,2026-07-17 12:30:00+00:00,0.00


## Status

`get_status()` and its per-dependency variants report the health of the API and its upstream providers.

In [18]:
status = client.get_status()
status

,storage,entso-e,elexon,epias
0,up,up,up,up


## Raw JSON access

`WattnetRawClient` exposes the exact same methods but returns the untouched JSON payload instead of a DataFrame — useful if you want to handle the nested structure yourself.

In [19]:
raw_client = WattnetRawClient(token=os.environ["WATTNET_TOKEN"])
zones_json = raw_client.get_zones()
print(type(zones_json), len(zones_json))
zones_json[0]

<class 'list'> 60


{'zone': 'AL',
 'full_name': 'Albania',
 'eic_code': '10YAL-KESH-----5',
 'country_code': 'ALB',
 'country_name': 'Albania',
 'provider': 'ENTSO-E',
 'neighbours': ['GR', 'ME', 'RS', 'XK']}

## Error handling

All errors derive from `wattnet.client.WattnetError`. Misconfiguring the client (e.g. no auth at all) raises `ConfigurationError` immediately, without making any request:

In [20]:
from wattnet.client import ConfigurationError

try:
    WattnetClient()
except ConfigurationError as exc:
    print(f"ConfigurationError: {exc}")

ConfigurationError: Must provide either `token`, or both `email` and `password`.
